In [1]:
!nvidia-smi -L 2>/dev/null || echo "No GPU (fine for this session)"
import sys; print("Python:", sys.version.split()[0])

GPU 0: Tesla T4 (UUID: GPU-0b9ae838-54e3-e23a-60d3-503e4e5883fd)
Python: 3.13.15


In [2]:
from pathlib import Path

WORK = Path("/content/drive/MyDrive/MiFO")
for sub in ["data/raw/fakenewsnet", "data/raw/liar", "data/processed", "notebooks"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

print("Workspace ready:", WORK)
print("Contents:", [p.name for p in WORK.iterdir()])

Workspace ready: /content/drive/MyDrive/MiFO
Contents: ['notebooks', 'data']


In [3]:
import hashlib, urllib.request

RAW = WORK / "data/raw/fakenewsnet"
BASE = "https://raw.githubusercontent.com/KaiDMML/FakeNewsNet/master/dataset"
EXPECTED = {  # first 16 hex of SHA256, from provenance record
    "politifact_fake.csv": "abe7fe7aad801b1e",
    "politifact_real.csv": "2500f86a7addca0f",
    "gossipcop_fake.csv":  "c6932bffadb1230b",
    "gossipcop_real.csv":  "d721e9a8b7e660da",
}

for name, want in EXPECTED.items():
    dest = RAW / name
    if not dest.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}", dest)
    got = hashlib.sha256(dest.read_bytes()).hexdigest()[:16]
    print(f"{'PASS' if got == want else 'FAIL'}  {name}  {got}")

PASS  politifact_fake.csv  abe7fe7aad801b1e
PASS  politifact_real.csv  2500f86a7addca0f
PASS  gossipcop_fake.csv  c6932bffadb1230b
PASS  gossipcop_real.csv  d721e9a8b7e660da


In [4]:
import hashlib, zipfile

url = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"
zpath = WORK / "data/raw/liar_dataset.zip"
if not zpath.exists():
    urllib.request.urlretrieve(url, zpath)

sha = hashlib.sha256(zpath.read_bytes()).hexdigest()
print("LIAR zip SHA256:", sha)          # <-- paste this to me
print("Size (MB):", round(zpath.stat().st_size / 1e6, 1))

with zipfile.ZipFile(zpath) as z:
    z.extractall(WORK / "data/raw/liar")
    print("Extracted:", z.namelist())

LIAR zip SHA256: 611c1addad919743dde15822b87a60bfb760d8f85597f25289e34621800654c7
Size (MB): 1.0
Extracted: ['README', 'test.tsv', 'train.tsv', 'valid.tsv']


In [5]:
import pandas as pd

COLS = ["json_id","statement_id","label","statement","subjects","speaker","job",
        "state","party","barely_true","false","half_true","mostly_true","pants_fire","context"]

liar = {}
for split in ["train", "valid", "test"]:
    df = pd.read_csv(WORK / f"data/raw/liar/{split}.tsv", sep="\t",
                     names=COLS, on_bad_lines="warn", quoting=3)
    liar[split] = df
    print(f"{split}: {df.shape[0]} rows x {df.shape[1]} cols")

print("\nLabel distribution (train):")
print(liar["train"]["label"].value_counts())
print("\nSample statement:", liar["train"]["statement"].iloc[0][:120])
print("Sample label:", liar["train"]["label"].iloc[0])

train: 10269 rows x 15 cols
valid: 1284 rows x 15 cols
test: 1283 rows x 15 cols

Label distribution (train):
label
On changing the rules for filibusters on presidential nominees                                                                                                                                                 3
During Sherrod Browns past decade as a D.C. politician, more than one out of every four jobs that has left America, left from Ohio. ... Sherrod Brown will own these horrendous Ohio job numbers next year.    2
"Obama says Iran is a 'tiny' country, 'doesn't pose a serious threat.'"                                                                                                                                        2
On support for the Export-Import Bank                                                                                                                                                                          2
Says Mitt Romney flip-flopped on abortion.      

In [6]:
import pandas as pd

COLS = ["json_id","label","statement","subjects","speaker","job","state","party",
        "barely_true","false","half_true","mostly_true","pants_fire","context"]  # 14, in true order

liar = {}
for split in ["train", "valid", "test"]:
    df = pd.read_csv(WORK / f"data/raw/liar/{split}.tsv", sep="\t",
                     names=COLS, quoting=3)
    liar[split] = df
    print(f"{split}: {df.shape[0]} rows x {df.shape[1]} cols")

print("\nLabel distribution (train):")
print(liar["train"]["label"].value_counts())
print("\nSample statement:", liar["train"]["statement"].iloc[0][:100])
print("Sample label:", liar["train"]["label"].iloc[0])
print("Sample context:", str(liar["train"]["context"].iloc[0])[:100])

train: 10269 rows x 14 cols
valid: 1284 rows x 14 cols
test: 1283 rows x 14 cols

Label distribution (train):
label
half-true      2123
false          1998
mostly-true    1966
true           1683
barely-true    1657
pants-fire      842
Name: count, dtype: int64

Sample statement: Says the Annies List political group supports third-trimester abortions on demand.
Sample label: false
Sample context: a mailer


In [7]:
import pandas as pd

RAW = WORK / "data/raw/fakenewsnet"

frames = []
for group in ["politifact", "gossipcop"]:
    for label in ["fake", "real"]:
        df = pd.read_csv(RAW / f"{group}_{label}.csv")
        df["source_group"] = group
        df["label_name"] = label
        df["label"] = 1 if label == "fake" else 0
        frames.append(df)
fn = pd.concat(frames, ignore_index=True)

# tweet_count: IDs are TAB-separated; NaN / empty / trailing tabs all -> correct count
fn["tweet_count"] = (fn["tweet_ids"].fillna("").astype(str)
                     .str.split("\t").map(lambda p: len([x for x in p if x.strip()])))

print("Total articles:", len(fn), "\n")
print(fn.groupby(["source_group", "label_name"]).agg(
    articles=("id", "size"),
    zero_tweets=("tweet_count", lambda s: int((s == 0).sum())),
    pct_zero_tweets=("tweet_count", lambda s: round(100 * (s == 0).mean(), 1)),
    median_tweets=("tweet_count", "median"),
).to_string())

Total articles: 23196 

                         articles  zero_tweets  pct_zero_tweets  median_tweets
source_group label_name                                                       
gossipcop    fake            5323          188              3.5           12.0
             real           16817         1058              6.3           45.0
politifact   fake             432           40              9.3           79.0
             real             624          215             34.5            8.0


In [11]:
from scipy.stats import mannwhitneyu

rows = []
for group in ["politifact", "gossipcop"]:
    for lab in ["fake", "real"]:
        s = fn[(fn.source_group == group) & (fn.label_name == lab)]["tweet_count"]
        rows.append({
            "group": group, "label": lab, "n": len(s),
            "mean": round(s.mean(), 1), "median": s.median(),
            "p25": s.quantile(.25), "p75": s.quantile(.75),
            "p90": s.quantile(.90), "p99": s.quantile(.99), "max": s.max(),
        })
dist = pd.DataFrame(rows)
print(dist.to_string(index=False), "\n")

for group in ["politifact", "gossipcop"]:
    a = fn[(fn.source_group == group) & (fn.label_name == "fake")]["tweet_count"]
    b = fn[(fn.source_group == group) & (fn.label_name == "real")]["tweet_count"]
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    r = round(2 * u / (len(a) * len(b)) - 1, 3)  # rank-biserial: +1 = fake higher
    verdict = "FAKE spreads more" if r > 0 else "REAL spreads more"
    print(f"{group:12s} U-test: p={p:.2e}  rank-biserial={r:+.3f}  -> {verdict}")

WORK = Path("C:/Users/anush/MiFO")
out = WORK / "data/"
out.mkdir(parents=True, exist_ok=True)
dist.to_csv(out / "tweet_distributions.csv", index=False)
print("\nSaved ->", out / "tweet_distributions.csv")

     group label     n  mean  median  p25   p75    p90      p99   max
politifact  fake   432 382.8    79.0 12.0 291.5  762.7  4755.89 29060
politifact  real   624 670.1     8.0  0.0 167.0 1540.2 12504.79 27377
 gossipcop  fake  5323 112.4    12.0  5.0  42.0  303.0  1039.00  2568
 gossipcop  real 16817  52.4    45.0 18.0  66.0   91.0   310.00  2082 

politifact   U-test: p=7.70e-14  rank-biserial=+0.268  -> FAKE spreads more
gossipcop    U-test: p=2.32e-193  rank-biserial=-0.269  -> REAL spreads more

Saved -> C:/Users/anush/MiFO/data/tweet_distributions.csv
